In [2]:
## 1. Reset stage

import shutil
from pathlib import Path
import duckdb

stage_root = Path("/home/jovyan/data/stage/handover_events")

if stage_root.exists():
    shutil.rmtree(stage_root)

stage_root.mkdir(parents=True, exist_ok=True)

print("Stage folder reset")


con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute("""
drop table handover_events
""").df()

print("dropped TABLE handover_events", flush=True)

Stage folder reset


CatalogException: Catalog Error: Table with name handover_events does not exist!
Did you mean "pg_sequences"?

In [3]:
## 2. Config

import csv
import gzip
from pathlib import Path
from datetime import datetime
import pandas as pd

RAW_DIR = Path("/home/jovyan/data/sim/raw/sim_test_decoded/2025/onefile")
STAGE_DIR = Path("/home/jovyan/data/stage/handover_events")
LOG_DIR = Path("/home/jovyan/data/logs")

LOG_DIR.mkdir(parents=True, exist_ok=True)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

# Column indexes (based on your sample)
IDX_EVENT_TS = 3
IDX_INGEST_TS = 3
IDX_PROCESSED_TS = 4
IDX_VEHICLE_ID = 5
IDX_PLMN = 17
IDX_CELL_ID = 18
IDX_RAT = 23
IDX_IMSI = 12

BATCH_SIZE = 100_000

In [4]:
#Environment Check

import os
import psutil

print("CPU count:", os.cpu_count())
print("RAM GB visible to container:", round(psutil.virtual_memory().total / (1024**3), 2))

CPU count: 24
RAM GB visible to container: 38.98


In [5]:
## Helper functions

def list_input_files(raw_dir: Path):
    exts = {".gz", ".csv", ".txt"}
    return sorted([p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts])

def open_text_file(path: Path):
    if path.suffix.lower() == ".gz":
        return gzip.open(path, mode="rt", newline="", encoding="utf-8", errors="replace")
    return open(path, mode="rt", newline="", encoding="utf-8", errors="replace")

def safe_get(row, idx):
    if idx >= len(row):
        return None
    value = row[idx].strip()
    return value if value != "" else None

def parse_timestamp(value):
    if not value:
        return None
    return pd.to_datetime(value, errors="coerce")

In [6]:
## Batch writer

def flush_batch(rows):
    if not rows:
        return

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["vehicle_id", "event_ts", "cell_id"])

    if df.empty:
        return

    df["event_date"] = df["event_ts"].dt.strftime("%Y-%m-%d")

    for event_date, part in df.groupby("event_date"):
        out_dir = STAGE_DIR / f"event_date={event_date}"
        out_dir.mkdir(parents=True, exist_ok=True)

        ts = datetime.utcnow().strftime("%Y%m%d%H%M%S%f")
        out_file = out_dir / f"stage_{ts}.parquet"
        part.to_parquet(out_file, index=False)

    del df

In [7]:
## File processing

def process_file(path: Path, max_rows=None):
    rows = []
    good = 0
    bad = 0

    with open_text_file(path) as f:
        reader = csv.reader(f)

        for i, row in enumerate(reader, start=1):
            if max_rows and i >= max_rows:
                break

            try:
                event_ts = parse_timestamp(safe_get(row, IDX_EVENT_TS))
                ingest_ts = parse_timestamp(safe_get(row, IDX_INGEST_TS))
                processed_ts = parse_timestamp(safe_get(row, IDX_PROCESSED_TS))
                vehicle_id = safe_get(row, IDX_VEHICLE_ID)
                imsi = safe_get(row, IDX_IMSI)
                plmn = safe_get(row, IDX_PLMN)
                cell_id = safe_get(row, IDX_CELL_ID)
                rat = safe_get(row, IDX_RAT)

                if pd.isna(event_ts) or not vehicle_id or not cell_id:
                    bad += 1
                    continue

                rows.append({
                    "vehicle_id": vehicle_id,
                    "imsi": imsi,
                    "event_ts": event_ts,
                    "ingest_ts": ingest_ts,
                    "processed_ts": processed_ts,
                    "cell_id": str(cell_id),
                    "plmn": plmn,
                    "rat": rat,
                    "source_file": str(path),
                })
                good += 1

                if len(rows) >= BATCH_SIZE:
                    flush_batch(rows)
                    rows = []

            except Exception:
                bad += 1

    flush_batch(rows)
    rows = []

    print(f"{path.name} → good={good:,} bad={bad:,}")

In [8]:
## 7. Validation

raw_files = list_input_files(RAW_DIR)

print(f"Found {len(raw_files)} file(s)")

# Test with first file, first 100k rows
process_file(raw_files[0], max_rows=100_000)

Found 1 files
2025_06_15.txt → good=99,971 bad=28


In [9]:
# one-file check

from pathlib import Path
import pandas as pd

stage_files = list(Path(STAGE_DIR).rglob("*.parquet"))
print("Stage files:", len(stage_files))

df = pd.read_parquet(stage_files[0])
df.head()

Stage files: 1


,vehicle_id,imsi,event_ts,ingest_ts,processed_ts,cell_id,plmn,rat,source_file,event_date
0,208BB19CFE3427967F0601DE3579DD7D,89011703322601638918,2025-06-15 10:20:37.318,2025-06-15 10:20:37.318,2025-06-15 10:20:47.045356,103064840,310-410-103064840,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15
1,2A7F8722C63B3C603E92E94CCB66755F,89011703322389429373,2025-06-15 10:20:37.709,2025-06-15 10:20:37.709,2025-06-15 10:20:47.045356,16422153,310-410-16422153,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15
2,3996B711E2C335B20BEDD6BDE111873B,89011703322644590951,2025-06-15 10:20:37.626,2025-06-15 10:20:37.626,2025-06-15 10:20:47.045356,180611863,310-410-180611863,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15
3,BADFA5BAF3AF25D7DC599D28CEA09097,89011703322251865522,2025-06-15 10:20:37.915,2025-06-15 10:20:37.915,2025-06-15 10:20:47.045356,201072138,310-410-201072138,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15
4,49D4001D4BA1DDF29B090FA414189315,89011703322367861571,2025-06-15 10:20:37.851,2025-06-15 10:20:37.851,2025-06-15 10:20:47.045356,175907405,310-410-175907405,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15


In [12]:
#multiple vehicles checks

from pathlib import Path
import pandas as pd

stage_files = list(Path(STAGE_DIR).rglob("*.parquet"))

sample = pd.concat([pd.read_parquet(p) for p in stage_files[:5]], ignore_index=True)

print("vehicles:", sample["vehicle_id"].nunique())
print(sample["vehicle_id"].value_counts().head())

vehicles: 17463
vehicle_id
A2FCC9AF8E480E683F655628A3E5E38D    14
C677F786DBCC6B3769A6725432950C42    14
548B68D5E573C637D32142E653183068    10
97951888DC9B69DB3332D265A722A880    10
6BCD2C7D9C544A61CEE0BF1DA38A4A99    10
Name: count, dtype: int64


In [13]:
## build results table

import duckdb
con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute(f"""
CREATE TABLE handover_events AS
SELECT *
FROM read_parquet('/home/jovyan/data/stage/handover_events/**/*.parquet')
""")

print("CREATED TABLE handover_events", flush=True)

CREATED TABLE handover_events


In [14]:
import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute("""
SELECT
    min(event_ts) as min_event,
    max(event_ts) as max_event,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT vehicle_id) AS vehicles,
    COUNT(DISTINCT cell_id) AS cells
FROM handover_events
""").df()



,min_event,max_event,total_rows,vehicles,cells
0,2025-06-15 10:20:31.694,2025-06-15 11:07:05.962,99971,17463,19580


In [15]:
import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")

con.execute("""
SELECT *
FROM handover_events
LIMIT 1
""").df()


,vehicle_id,imsi,event_ts,ingest_ts,processed_ts,cell_id,plmn,rat,source_file,event_date
0,208BB19CFE3427967F0601DE3579DD7D,89011703322601638918,2025-06-15 10:20:37.318,2025-06-15 10:20:37.318,2025-06-15 10:20:47.045356,103064840,310-410-103064840,4G,/home/jovyan/data/sim/raw/sim_test_decoded/202...,2025-06-15


In [16]:
import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")
con.execute("""
SELECT COUNT(DISTINCT plmn)  -- or test IMSI column if added
FROM handover_events
""").df()

,count(DISTINCT plmn)
0,19580


In [17]:
import csv

sample_vals = {}

for idx in range(30):  # adjust range if needed
    sample_vals[idx] = set()

for f in raw_files[:3]:
    with open_text_file(f) as fh:
        reader = csv.reader(fh)
        for i, row in enumerate(reader):
            if i > 50000:
                break
            for idx in sample_vals:
                if idx < len(row):
                    sample_vals[idx].add(row[idx])

for idx, vals in sample_vals.items():
    print(idx, len(vals))

0 49967
1 3
2 15525
3 48968
4 1461
5 17422
6 2
7 1
8 710
9 1
10 3
11 6
12 17422
13 1
14 7
15 7
16 6
17 17700
18 17695
19 7
20 12
21 6
22 1477
23 2
24 43562
25 35
26 35
27 472
28 83
29 19


In [18]:
import csv

for idx in [5, 12, 14]:
    vals = set()
    for f in raw_files[:10]:
        with open_text_file(f) as fh:
            reader = csv.reader(fh)
            for i, row in enumerate(reader):
                if i > 100000:
                    break
                if idx < len(row):
                    vals.add(row[idx])
    print(idx, len(vals))

5 17467
12 17467
14 7


In [19]:
# after loading col5 as candidate_id

import duckdb

con = duckdb.connect("/home/jovyan/data/analytics.duckdb")
#con.execute("DESCRIBE handover_events").df()
con.execute("select count(1) from handover_events").df()


,count(1)
0,99971


In [25]:
import csv

raw_sim_ids = set()
row_count = 0

with open_text_file(raw_files[0]) as f:
    reader = csv.reader(f)
    for row in reader:
        row_count += 1
        sim_id = safe_get(row, IDX_VEHICLE_ID)
        if sim_id:
            raw_sim_ids.add(sim_id)

print("Raw rows:", row_count)
print("Distinct SIM_ID in raw file:", len(raw_sim_ids))
print("Sample SIM_IDs:", list(raw_sim_ids)[:10])

Raw rows: 3119435
Distinct SIM_ID in raw file: 18594
Sample SIM_IDs: ['3D0E2C9607D427401FB0AD9CBAD59342', 'EEE1703081F29051EEA9D8FDCAC1C700', '3D2C4717F14208511F0892AC2D435495', 'C48FE1DF7B12ACD90011EBF16B76649E', '46D4FF50848E0E4B57D9ED24C1907026', 'FD22D74E8F92F86F690E38BB1BC5ED09', '42277316674A42FBF9ACC7BEBE052290', '7CA67F0F3F5A7E1A659CA1F0409EA596', 'F140A413CDBACB241431A200B94FD8E9', '162A3D608744FA10DAD4190AA00DF27C']
